In [26]:
"""
Computes an Urban Accessibility Index for Marseille, France, combining
four urban indicators — schools, subway stations, bike paths, and parks —
at two geographic scales:
  1. Administrative districts (arrondissements)
  2. H3 hexagonal grid cells (resolution 9, ~105 m edge length)
"""

'\nComputes an Urban Accessibility Index for Marseille, France, combining\nfour urban indicators — schools, subway stations, bike paths, and parks —\nat two geographic scales:\n  1. Administrative districts (arrondissements)\n  2. H3 hexagonal grid cells (resolution 9, ~105 m edge length)\n'

In [27]:
# =============================================================================
# 1. IMPORTS
# =============================================================================

import geopandas as gpd
import h3pandas
import h3
import leafmap
from shapely.geometry import Point, Polygon
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd
import osmnx as ox

In [28]:
# =============================================================================
# 2. DATA COLLECTION — OpenStreetMap features
# =============================================================================

In [29]:
place = 'Marseille, France'

# Fetch urban features from OSM using standard tags.
# Each call returns a GeoDataFrame with mixed geometry types (points, lines, polygons).
schools = ox.features_from_place(place, tags={'amenity':'school'})
subways = ox.features_from_place(place, tags={'railway':'subway'})
bike_paths = ox.features_from_place(place, tags={'highway':'cycleway'})
parks = ox.features_from_place(place, tags={'leisure':'park'})

In [30]:
# Build French ordinal suffixes (1er, 2e, 3e, ..., 16e)
district_numbers = [f"{i}{'er' if i < 2 else 'e'}" for i in range(1, 17)]
boundaries_list = []

for dn in district_numbers:
     place = dn + ' arrondissement, Marseille'
     admin_d = ox.geocode_to_gdf(place)  # Returns the administrative boundary polygon
     boundaries_list.append(admin_d)

# Concatenate all district boundaries into a single GeoDataFrame
districts = pd.concat(boundaries_list)
districts = districts.reset_index()

In [31]:
# Reprojecting all data in Lambert93 before any distance or area measurement.
schools = schools.to_crs(2154)
subways = subways.to_crs(2154)
bike_paths = bike_paths.to_crs(2154)
districts = districts.to_crs(2154)
parks = parks.to_crs(2154)

In [32]:
# =============================================================================
# 5. DISTRICT-LEVEL ANALYSIS — Count/measure features per arrondissement
# =============================================================================

In [33]:
def analyze_district(district_geometry):
    """
    For a given district polygon, compute four accessibility indicators:
      - num_schools       : count of school features intersecting the district
      - num_subways       : count of subway station features intersecting the district
      - bike_path_length  : total length (m) of cycleways intersecting the district
      - park_area         : total area (m²) of parks intersecting the district
   
    Returns
    -------
    tuple : (int, int, float, float)
    """
    num_schools = schools[schools.geometry.intersects(district_geometry)].shape[0]
    num_subways = subways[subways.geometry.intersects(district_geometry)].shape[0]
    bike_path_length = bike_paths[bike_paths.geometry.intersects(district_geometry)].length.sum()
    park_area = parks[parks.geometry.intersects(district_geometry)].area.sum()

    return num_schools, num_subways, bike_path_length, park_area

In [34]:
# Apply the analysis function to every district
districts[['num_schools', 'num_subways', 'bike_path_length', 'park_area']] = districts.geometry.apply(
    lambda geom: pd.Series(analyze_district(geom))
)

In [35]:
# =============================================================================
# 6. NORMALIZATION (first pass) — Min-Max scaling to [0, 1]
# =============================================================================

# Ensures all four indicators are on the same scale before aggregation.
# Without this step, large raw values (e.g., park area in m²) would dominate.
scaler = MinMaxScaler()
columns_to_normalize = ['num_schools', 'num_subways', 'bike_path_length', 'park_area']
districts[columns_to_normalize] = scaler.fit_transform(districts[columns_to_normalize])

In [ ]:
# =============================================================================
# 7. SPATIAL SMOOTHING — Average each district with its touching neighbors
# =============================================================================

In [36]:
# Aggregate results using touching neighborhoods
def aggregate_touching_districts(district_index):
    """
    Spatially smooth indicator values by averaging a district's values
    with those of its directly adjacent (touching) districts.
    If a district has no neighbors (isolated geometry), its own values are kept.
 
    Returns
    -------
    pd.Series : averaged indicator values for the four columns
    """
    current_geometry = districts.loc[district_index, 'geometry']
    # .touches() returns True for districts sharing a boundary edge or point
    touching_indices = districts[districts.geometry.touches(current_geometry)].index

    if not touching_indices.empty:
        district_values = districts.loc[touching_indices, columns_to_normalize].mean()
    else:
        district_values = districts.loc[neighborhood_index, columns_to_normalize]

    return district_values

In [37]:
# Apply aggregation to each neighborhood
districts[columns_to_normalize] = districts.index.to_series().apply(
    lambda idx: aggregate_touching_districts(idx)
)

In [38]:
# =============================================================================
# 8. NORMALIZATION (second pass) — Re-normalize after smoothing
# =============================================================================

# Smoothing changes the value range --> Final normalization (to restore the 0 to 1 scale) 
districts[columns_to_normalize] = scaler.fit_transform(districts[columns_to_normalize])

In [39]:
# =============================================================================
# 9. COMPOSITE SCORE — Sum of the four normalized indicators
# =============================================================================

# Score range: 0 (lowest accessibility) to 4 (highest accessibility)
districts['index_score'] = districts['num_schools'] + districts['num_subways'] + districts['bike_path_length'] + districts['park_area'] 

In [40]:
# =============================================================================
# 10. DISTRICT-LEVEL MAP — Visualize with leafmap
# =============================================================================
# Quantile classification ensures equal counts per class regardless of value skew.
# The 'Blues' colormap encodes higher scores with darker shades.

import leafmap

m = leafmap.Map()
m.add_data(
    districts, column="index_score", scheme="Quantiles", cmap="Blues", legend_title="Index"
)
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

In [52]:
# =============================================================================
# 11. H3 HEXAGONAL GRID — Polyfill districts with H3 cells (resolution 9)
# =============================================================================
# H3 resolution 9 produces hexagons with ~105 m edge length (~0.1 km²).

In [41]:
# H3 requires WGS84 (EPSG:4326) coordinates — reproject before polyfilling.
districts = districts.to_crs('EPSG:4326')

In [42]:
# .polyfill() fills each district polygon with H3 cell indices
# explode=True creates one row per H3 cell
resolution = 9  # Adjust resolution as needed
gdf_h3 = districts.h3.polyfill(resolution, explode=True)

In [ ]:
# =============================================================================
# 12. H3 GRID CLEANUP — Remove null cells and set H3 index
# =============================================================================

In [43]:
gdf_h3 = gdf_h3[gdf_h3['h3_polyfill'].isnull() == False].set_index('h3_polyfill')
gdf_h3.index.name = None # Clean up the index name for readability

In [44]:
# Convert H3 cell indices to actual hexagonal boundary geometries
gdf_h3 = gdf_h3.h3.h3_to_geo_boundary()

In [45]:
# Reproject to Lambert-93 for metric spatial analysis
gdf_h3_proj = gdf_h3.to_crs(2154)

In [46]:
# =============================================================================
# 13. HEX-LEVEL ANALYSIS — Buffer-based accessibility measurement
# =============================================================================
# Unlike the district analysis (simple intersection), the hex-level analysis
# uses circular buffers centered on each cell to capture nearby features:
#   - 1600 m ≈ 20-min walk / 5-min bike: suitable for schools, transit, cycle paths
#   - 800 m  ≈ 10-min walk: suitable for parks (shorter acceptable distance)

def analyze_access(hex_geometry):
    """
    Measure accessibility indicators for a single H3 hexagon using spatial buffers.
 
    Returns
    -------
    tuple : (int, int, float, float)
    """
    buffer_1600m = hex_geometry.buffer(1600) # ~20-min walk radius
    buffer_800m = hex_geometry.buffer(800) # ~10-min walk radius

    # Count features within buffers
    num_schools = schools[schools.geometry.intersects(buffer_1600m)].shape[0]
    num_subways = subways[subways.geometry.intersects(buffer_1600m)].shape[0]
    bike_path_length = bike_paths[bike_paths.geometry.intersects(buffer_1600m)].length.sum()
    park_area = parks[parks.geometry.intersects(buffer_800m)].area.sum()

    return num_schools, num_subways, bike_path_length, park_area

In [47]:
# Apply analysis to each hex cell
gdf_h3_proj[['num_schools', 'num_subways', 'bike_path_length', 'park_area']] = gdf_h3_proj.geometry.apply(
    lambda hex_geom: pd.Series(analyze_access(hex_geom))
)

In [48]:
# =============================================================================
# 14. STORE H3 INDEX AS COLUMN — Required for neighbor lookup
# =============================================================================
# The H3 cell ID (used by h3.grid_disk) is stored as a regular column
# so it can be referenced inside the aggregation function below.

gdf_h3_proj['h3_index'] = gdf_h3_proj.index

In [ ]:
# =============================================================================
# 15. HEX-LEVEL NORMALIZATION + SPATIAL SMOOTHING WITH H3 NEIGHBORS
# =============================================================================

In [49]:
# Normalize results
scaler = MinMaxScaler()
normalized_columns = ['num_schools', 'num_subways', 'bike_path_length', 'park_area']
gdf_h3_proj[normalized_columns] = scaler.fit_transform(gdf_h3_proj[normalized_columns])

# Aggregate results using neighboring cells
def aggregate_neighbors(h3_index):
    """
    Smooth a hex cell's indicator values by averaging over its 2-ring neighborhood.
    h3.grid_disk(index, k) returns all H3 cells within k rings of the given cell,
    including the cell itself (total up to 1 + 6 + 12 = 19 cells for k=2).
 
    Returns
    -------
    pd.Series : mean indicator values across the neighborhood
    """
    neighbors = h3.grid_disk(h3_index, 2)  # 2-k ring
    neighbor_values = gdf_h3_proj[gdf_h3_proj['h3_index'].isin(neighbors)][normalized_columns].mean()
    return neighbor_values

# Apply neighbor-based spatial smoothing to all hex cells
gdf_h3_proj[normalized_columns] = gdf_h3_proj['h3_index'].apply(
    lambda h3_index: aggregate_neighbors(h3_index)
)

# Final normalized analysis
gdf_h3_proj[normalized_columns] = scaler.fit_transform(gdf_h3_proj[normalized_columns])

# Save or visualize the results
gdf_h3_proj.to_file("access_index.geojson", driver="GeoJSON")

In [50]:
# =============================================================================
# 16. COMPOSITE SCORE + FINAL MAP — Hex-level visualization
# =============================================================================

gdf_h3_proj['index_score'] = gdf_h3_proj['num_schools'] + gdf_h3_proj['num_subways'] + gdf_h3_proj['bike_path_length'] + gdf_h3_proj['park_area'] 

In [51]:
m = leafmap.Map()
m.add_data(
    gdf_h3_proj, column="index_score", scheme="Quantiles", cmap="Blues", legend_title="Index"
)
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…